# Phase 1: Data Loading & Inspection

### Step 1: Data Loading

In [46]:
import pandas as pd
import numpy as np
from IPython.display import display

# 1. Load the data 
# Using "../" goes up one directory level to find the "data" folder
file_path = "../data/risk_factors_cervical_cancer.csv"

df = pd.read_csv(file_path)

# 2. Check the shape
rows, cols = df.shape
print(f"Dataset Shape: {rows} patients and {cols} features\n")

# Peek at the first few rows
display(df.head())

Dataset Shape: 858 patients and 36 features



,Age,Number of sexual partners,First sexual intercourse,Num of pregnancies,Smokes,Smokes (years),Smokes (packs/year),Hormonal Contraceptives,Hormonal Contraceptives (years),IUD,...,STDs: Time since first diagnosis,STDs: Time since last diagnosis,Dx:Cancer,Dx:CIN,Dx:HPV,Dx,Hinselmann,Schiller,Citology,Biopsy
0,18,4.0,15.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
1,15,1.0,14.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
2,34,1.0,?,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
3,52,5.0,16.0,4.0,1.0,37.0,37.0,1.0,3.0,0.0,...,?,?,1,0,1,0,0,0,0,0
4,46,3.0,21.0,4.0,0.0,0.0,0.0,1.0,15.0,0.0,...,?,?,0,0,0,0,0,0,0,0


### Step 2: Initial Diagnostic Checks

In [47]:
# 3. Inspect Data Types
print("--- DataFrame Info ---")
df.info()
print("\n")

# 4. Investigate Suspicious 'Object' Columns
suspicious_col = 'Number of sexual partners' 
print(f"--- Unique values in '{suspicious_col}' ---")
print(df[suspicious_col].unique()) 
print("\n")

# 5. Identify the Target Variable
target = 'Biopsy'
print(f"--- Target Variable Distribution ({target}) ---")
display(df[target].value_counts(normalize=True).round(4) * 100)

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 858 entries, 0 to 857
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   Age                                 858 non-null    int64 
 1   Number of sexual partners           858 non-null    object
 2   First sexual intercourse            858 non-null    object
 3   Num of pregnancies                  858 non-null    object
 4   Smokes                              858 non-null    object
 5   Smokes (years)                      858 non-null    object
 6   Smokes (packs/year)                 858 non-null    object
 7   Hormonal Contraceptives             858 non-null    object
 8   Hormonal Contraceptives (years)     858 non-null    object
 9   IUD                                 858 non-null    object
 10  IUD (years)                         858 non-null    object
 11  STDs                               

Biopsy
0    93.59
1     6.41
Name: proportion, dtype: float64

# Phase 2: Data Cleaning & Type Conversion

In [48]:
# 1. Replace the text '?' with standard numpy NaN
df = df.replace('?', np.nan)

# 2. Convert all columns to numeric (Future-Proof Version)
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except ValueError:
        pass

print("Data types successfully converted!\n")

print("--- Updated DataFrame Info ---")
df.info()

Data types successfully converted!

--- Updated DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 858 entries, 0 to 857
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Age                                 858 non-null    int64  
 1   Number of sexual partners           832 non-null    float64
 2   First sexual intercourse            851 non-null    float64
 3   Num of pregnancies                  802 non-null    float64
 4   Smokes                              845 non-null    float64
 5   Smokes (years)                      845 non-null    float64
 6   Smokes (packs/year)                 845 non-null    float64
 7   Hormonal Contraceptives             750 non-null    float64
 8   Hormonal Contraceptives (years)     750 non-null    float64
 9   IUD                                 741 non-null    float64
 10  IUD (years)                         741 non

# Phase 3: Data Assessment & Investigation

### Step 1: Consistency check

In [49]:
# Cell
print("==========================================")
print("PHASE 3: DATA ASSESSMENT & INVESTIGATION")
print("==========================================")
print("--- Step 1: Logical Consistency Checks ---\n")

print("Note: Row IDs refer to the pandas DataFrame Index.")
print("To find the raw CSV line number, calculate: Row ID + 2\n")

# ---------------------------------------------------------
# TEST 1: The "Smokes" Logic
# ---------------------------------------------------------
print("TEST 1: 'Smokes' Consistency")
smokes_flaws = df[
    ((df['Smokes'] == 0) | (df['Smokes'].isnull())) & 
    ((df['Smokes (years)'] > 0) | (df['Smokes (packs/year)'] > 0))
]

print(f"Found {len(smokes_flaws)} patients with contradictory 'Smokes' data.")
if len(smokes_flaws) > 0:
    print(f"Row IDs: {smokes_flaws.index.tolist()}")
    print("Example of the flawed data:")
    print(smokes_flaws[['Smokes', 'Smokes (years)', 'Smokes (packs/year)']].head())
print("-" * 40 + "\n")

# ---------------------------------------------------------
# TEST 2: The "STDs" Logic
# ---------------------------------------------------------
print("TEST 2: 'STDs' Consistency")
all_std_related_cols = [col for col in df.columns if col.startswith('STDs') and col != 'STDs']

stds_flaws = df[
    ((df['STDs'] == 0) | (df['STDs'].isnull())) & 
    (df[all_std_related_cols].sum(axis=1) > 0)
]

print(f"Found {len(stds_flaws)} patients with contradictory 'STDs' data.")
if len(stds_flaws) > 0:
    print(f"Row IDs: {stds_flaws.index.tolist()}")
    print("Example of the flawed data:")
    cols_to_show = ['STDs'] + all_std_related_cols[:4]
    print(stds_flaws[cols_to_show].head())
print("-" * 40 + "\n")

# ---------------------------------------------------------
# TEST 3: The "Reverse" Logic
# ---------------------------------------------------------
print("TEST 3: Reverse Logic (0s should mean 0s)")
smokes_reverse_flaws = df[(df['Smokes'] == 0) & ((df['Smokes (years)'].isnull()) | (df['Smokes (packs/year)'].isnull()))]
print(f"Found {len(smokes_reverse_flaws)} patients who are non-smokers (0) but have NaN in years/packs.")
if len(smokes_reverse_flaws) > 0:
    print(f"Row IDs: {smokes_reverse_flaws.index.tolist()}")

stds_reverse_flaws = df[(df['STDs'] == 0) & (df['STDs (number)'].isnull())]
print(f"Found {len(stds_reverse_flaws)} patients who have NO STDs (0) but have NaN in STDs (number).")
if len(stds_reverse_flaws) > 0:
    print(f"Row IDs: {stds_reverse_flaws.index.tolist()}")
print("-" * 40 + "\n")

# ---------------------------------------------------------
# TEST 4: The "Condylomatosis" Bidirectional Logic
# ---------------------------------------------------------
print("TEST 4: 'Condylomatosis' Sub-feature Consistency")
condy_cols = [col for col in df.columns if 'condylomatosis' in col and col != 'STDs:condylomatosis']

condy_flaws = df[
    ((df['STDs:condylomatosis'] == 0) | (df['STDs:condylomatosis'].isnull())) & 
    (df[condy_cols].sum(axis=1) > 0)
]

print(f"Found {len(condy_flaws)} patients with contradictory 'condylomatosis' data.")
if len(condy_flaws) > 0:
    print(f"Row IDs: {condy_flaws.index.tolist()}")
    print("Example of the flawed data:")
    cols_to_show = ['STDs:condylomatosis'] + condy_cols
    print(condy_flaws[cols_to_show].head())
print("-" * 40 + "\n")

# ---------------------------------------------------------
# TEST 5: The "Condylomatosis" Reverse Logic
# ---------------------------------------------------------
print("TEST 5: 'Condylomatosis' Reverse Logic (0s should mean 0s)")
condy_reverse_flaws = df[
    (df['STDs:condylomatosis'] == 0) & 
    (df[condy_cols].isnull().any(axis=1))
]

print(f"Found {len(condy_reverse_flaws)} patients with NO condylomatosis (0) but missing anatomical data.")
if len(condy_reverse_flaws) > 0:
    print(f"Row IDs: {condy_reverse_flaws.index.tolist()}")
    print("Example of the flawed data:")
    print(condy_reverse_flaws[['STDs:condylomatosis'] + condy_cols].head())
print("-" * 40 + "\n")

# ---------------------------------------------------------
# TEST 6: The "Hormonal Contraceptives" Logic
# ---------------------------------------------------------
print("TEST 6: 'Hormonal Contraceptives' Consistency")
hc_flaws = df[
    ((df['Hormonal Contraceptives'] == 0) | (df['Hormonal Contraceptives'].isnull())) & 
    (df['Hormonal Contraceptives (years)'] > 0)
]

print(f"Found {len(hc_flaws)} patients with contradictory 'Hormonal Contraceptives' data.")
if len(hc_flaws) > 0:
    print(f"Row IDs: {hc_flaws.index.tolist()}")
    print("Example of the flawed data:")
    print(hc_flaws[['Hormonal Contraceptives', 'Hormonal Contraceptives (years)']].head())
print("-" * 40 + "\n")

# ---------------------------------------------------------
# TEST 7: The "Hormonal Contraceptives" Reverse Logic
# ---------------------------------------------------------
print("TEST 7: 'Hormonal Contraceptives' Reverse Logic")
hc_reverse_flaws = df[
    (df['Hormonal Contraceptives'] == 0) & 
    (df['Hormonal Contraceptives (years)'].isnull())
]

print(f"Found {len(hc_reverse_flaws)} patients with NO Hormonal Contraceptives (0) but NaN in years.")
if len(hc_reverse_flaws) > 0:
    print(f"Row IDs: {hc_reverse_flaws.index.tolist()}")
    print("Example of the flawed data:")
    print(hc_reverse_flaws[['Hormonal Contraceptives', 'Hormonal Contraceptives (years)']].head())
print("-" * 40 + "\n")

# ---------------------------------------------------------
# TEST 8: The "IUD" Logic
# ---------------------------------------------------------
print("TEST 8: 'IUD' Consistency")
iud_flaws = df[
    ((df['IUD'] == 0) | (df['IUD'].isnull())) & 
    (df['IUD (years)'] > 0)
]

print(f"Found {len(iud_flaws)} patients with contradictory 'IUD' data.")
if len(iud_flaws) > 0:
    print(f"Row IDs: {iud_flaws.index.tolist()}")
    print("Example of the flawed data:")
    print(iud_flaws[['IUD', 'IUD (years)']].head())
print("-" * 40 + "\n")

# ---------------------------------------------------------
# TEST 9: The "IUD" Reverse Logic
# ---------------------------------------------------------
print("TEST 9: 'IUD' Reverse Logic")
iud_reverse_flaws = df[
    (df['IUD'] == 0) & 
    (df['IUD (years)'].isnull())
]

print(f"Found {len(iud_reverse_flaws)} patients with NO IUD (0) but NaN in years.")
if len(iud_reverse_flaws) > 0:
    print(f"Row IDs: {iud_reverse_flaws.index.tolist()}")
    print("Example of the flawed data:")
    print(iud_reverse_flaws[['IUD', 'IUD (years)']].head())
print("-" * 40 + "\n")

# ---------------------------------------------------------
# TEST 10: Chronological Boundaries
# ---------------------------------------------------------
print("TEST 10: Chronological Age Constraints")
bad_fsi = df['First sexual intercourse'] > df['Age']
bad_smokes_years = df['Smokes (years)'] >= df['Age']
bad_hc_years = df['Hormonal Contraceptives (years)'] >= df['Age']
bad_iud_years = df['IUD (years)'] >= df['Age']

chrono_flaws = df[bad_fsi | bad_smokes_years | bad_hc_years | bad_iud_years]

print(f"Found {len(chrono_flaws)} patients violating chronological time limits.")
if len(chrono_flaws) > 0:
    print(f"Row IDs: {chrono_flaws.index.tolist()}")
    print("Example of the flawed data:")
    cols = ['Age', 'First sexual intercourse', 'Smokes (years)', 'Hormonal Contraceptives (years)', 'IUD (years)']
    print(chrono_flaws[cols].head())
print("-" * 40 + "\n")

# ---------------------------------------------------------
# TEST 11: STD Diagnosis Continuity (Hardened Version)
# ---------------------------------------------------------
print("TEST 11: STD Diagnosis Continuity")
try:
    col_num_diag = [c for c in df.columns if 'Number of diagnosis' in c][0]
    col_first_diag = [c for c in df.columns if 'first diagnosis' in c][0]
    col_last_diag = [c for c in df.columns if 'last diagnosis' in c][0]
except IndexError:
    raise ValueError("CRITICAL DATA ERROR: One or more 'diagnosis' columns are missing.")

bad_std_count = ((df['STDs'] == 0) | df['STDs'].isnull()) & (df[col_num_diag] > 0)
bad_std_count_reverse = (df['STDs'] == 0) & df[col_num_diag].isnull()
time_paradox = df[col_first_diag] < df[col_last_diag]

std_timeline_flaws = df[bad_std_count | bad_std_count_reverse | time_paradox]

print(f"Found {len(std_timeline_flaws)} patients with STD timeline or count contradictions.")
if len(std_timeline_flaws) > 0:
    print(f"Row IDs: {std_timeline_flaws.index.tolist()}")
    print("Example of the flawed data:")
    cols = ['STDs', col_num_diag, col_first_diag, col_last_diag]
    print(std_timeline_flaws[cols].head())
print("-" * 40 + "\n")

PHASE 3: DATA ASSESSMENT & INVESTIGATION
--- Step 1: Logical Consistency Checks ---

Note: Row IDs refer to the pandas DataFrame Index.
To find the raw CSV line number, calculate: Row ID + 2

TEST 1: 'Smokes' Consistency
Found 0 patients with contradictory 'Smokes' data.
----------------------------------------

TEST 2: 'STDs' Consistency
Found 0 patients with contradictory 'STDs' data.
----------------------------------------

TEST 3: Reverse Logic (0s should mean 0s)
Found 0 patients who are non-smokers (0) but have NaN in years/packs.
Found 0 patients who have NO STDs (0) but have NaN in STDs (number).
----------------------------------------

TEST 4: 'Condylomatosis' Sub-feature Consistency
Found 0 patients with contradictory 'condylomatosis' data.
----------------------------------------

TEST 5: 'Condylomatosis' Reverse Logic (0s should mean 0s)
Found 0 patients with NO condylomatosis (0) but missing anatomical data.
----------------------------------------

TEST 6: 'Hormonal Con

### Step 1.b: Drop the corrupted rows

In [50]:
# Drop the fundamentally corrupted rows identified in Test 10
df = df.drop(index=[312, 812])

# Optional but recommended: Reset the index after dropping rows to maintain a clean sequence
df = df.reset_index(drop=True)

### Step 2: Missing data handling

In [51]:
print("--- Step 2: Missing Data Assessment ---")

# Calculate the percentage of missing values per column
# df.isnull().sum() counts the NaNs, dividing by len(df) gets the ratio, * 100 gets percentage
missing_percentages = (df.isnull().sum() / len(df)) * 100

# Filter to only show columns that have AT LEAST SOME missing data (> 0%)
missing_cols = missing_percentages[missing_percentages > 0].sort_values(ascending=False)

print(f"Total columns with missing data: {len(missing_cols)} out of {df.shape[1]}\n")

if len(missing_cols) > 0:
    print("--- SEVERE Missing Data (> 50%) [Drop Candidates] ---")
    severe_cols = missing_cols[missing_cols > 50]
    if len(severe_cols) > 0:
        print(severe_cols.round(2).astype(str) + ' %')
    else:
        print("None")
        
    print("\n--- MODERATE/MINOR Missing Data (<= 50%) [Imputation Candidates] ---")
    moderate_cols = missing_cols[missing_cols <= 50]
    if len(moderate_cols) > 0:
        print(moderate_cols.round(2).astype(str) + ' %')
    else:
        print("None")
print("\n" + "="*40 + "\n")

--- Step 2: Missing Data Assessment ---
Total columns with missing data: 26 out of 36

--- SEVERE Missing Data (> 50%) [Drop Candidates] ---
STDs: Time since first diagnosis    91.71 %
STDs: Time since last diagnosis     91.71 %
dtype: object

--- MODERATE/MINOR Missing Data (<= 50%) [Imputation Candidates] ---
IUD (years)                           13.55 %
IUD                                   13.55 %
Hormonal Contraceptives               12.62 %
Hormonal Contraceptives (years)       12.62 %
STDs (number)                         12.27 %
STDs                                  12.27 %
STDs:vulvo-perineal condylomatosis    12.27 %
STDs:vaginal condylomatosis           12.27 %
STDs:pelvic inflammatory disease      12.27 %
STDs:syphilis                         12.27 %
STDs:genital herpes                   12.27 %
STDs:molluscum contagiosum            12.27 %
STDs:HIV                              12.27 %
STDs:AIDS                             12.27 %
STDs:condylomatosis                   12.27

# Phase 4: Handling Missing Data (Imputation)

### Option A: Imputing with median

In [52]:
# print("==========================================")
# print("PHASE 4: HANDLING MISSING DATA (IMPUTATION)")
# print("==========================================")

# # ---------------------------------------------------------
# # 1. Drop the "Not Applicable" Columns
# # ---------------------------------------------------------
# cols_to_drop = ['STDs: Time since first diagnosis', 'STDs: Time since last diagnosis']
# df = df.drop(columns=cols_to_drop, errors='ignore')
# print(f"Dropped {len(cols_to_drop)} columns due to >90% missing data.\n")

# # ---------------------------------------------------------
# # 2. Impute STDs with the Median
# # ---------------------------------------------------------
# print("--- STD Columns Imputation Log ---")
# std_cols = [col for col in df.columns if col.startswith('STDs')]

# imputed_count = 0
# for col in std_cols:
#     # Only calculate and print if the column actually has missing data
#     if df[col].isnull().sum() > 0:
#         col_median = df[col].median()
#         df[col] = df[col].fillna(col_median)
#         print(f"'{col}' missing values filled with its median: {col_median}")
#         imputed_count += 1

# print(f"Finished imputing {imputed_count} STD-related columns.\n")

# # ---------------------------------------------------------
# # 3. Conditional Imputation (Contraceptives & Smokes)
# # ---------------------------------------------------------
# print("--- Conditional Imputation Log ---")
# # We structure this as a list of (Master Column, [List of Sub-Columns])
# conditional_groups = [
#     ('IUD', ['IUD (years)']), 
#     ('Hormonal Contraceptives', ['Hormonal Contraceptives (years)']),
#     ('Smokes', ['Smokes (years)', 'Smokes (packs/year)'])
# ]

# for master, sub_cols in conditional_groups:
#     # Step A: Fill the master categorical column with its median (0 or 1)
#     master_median = df[master].median()
#     df[master] = df[master].fillna(master_median)
#     print(f"Master '{master}' missing values filled with: {master_median}")
    
#     for sub in sub_cols:
#         # Step B: Calculate the median strictly for patients who used it (> 0)
#         user_median = df[df[master] == 1][sub].median()
        
#         # Step C: Apply conditional logic to fill the missing sub-columns
#         df.loc[df[master] == 0, sub] = df.loc[df[master] == 0, sub].fillna(0)
#         df.loc[df[master] == 1, sub] = df.loc[df[master] == 1, sub].fillna(user_median)
        
#         print(f"  -> '{sub}' missing values conditionally filled:")
#         print(f"     - With 0.0 if '{master}' == 0")
#         print(f"     - With {user_median} if '{master}' == 1")
# print("\n")

# # ---------------------------------------------------------
# # 4. Global Fallback for remaining columns
# # ---------------------------------------------------------
# print("--- Standard Imputation Log (Global Fallback) ---")
# remaining_imputed = 0
# for col in df.columns:
#     if df[col].isnull().sum() > 0:
#         col_median = df[col].median()
#         df[col] = df[col].fillna(col_median)
#         print(f"'{col}' missing values filled with: {col_median}")
#         remaining_imputed += 1

# if remaining_imputed == 0:
#     print("No remaining columns needed fallback imputation.")

# # ---------------------------------------------------------
# # 5. Final Proof
# # ---------------------------------------------------------
# print("\n--- Final Dataset Check ---")
# print(f"Total missing values left in the ENTIRE dataset: {df.isnull().sum().sum()}")
# print(f"Final Dataset Shape: {df.shape[0]} patients, {df.shape[1]} features")
# print("==========================================\n")

### Option B: Imputing with KNN

In [53]:
# Cell
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler

print("==========================================")
print("PHASE 4: HANDLING MISSING DATA (IMPUTATION)")
print("==========================================")

# ---------------------------------------------------------
# 1. Drop the "Not Applicable" Columns
# ---------------------------------------------------------
cols_to_drop = ['STDs: Time since first diagnosis', 'STDs: Time since last diagnosis']
df = df.drop(columns=cols_to_drop, errors='ignore')
print(f"Dropped {len(cols_to_drop)} columns due to >90% missing data.\n")

# ---------------------------------------------------------
# 2. Deterministic Conditional Imputation (Contraceptives & Smokes)
# Executed FIRST to provide a denser matrix for KNN
# ---------------------------------------------------------
print("--- Deterministic Conditional Imputation Log ---")
conditional_groups = [
    ('IUD', ['IUD (years)']), 
    ('Hormonal Contraceptives', ['Hormonal Contraceptives (years)']),
    ('Smokes', ['Smokes (years)', 'Smokes (packs/year)'])
]

for master, sub_cols in conditional_groups:
    master_median = df[master].median()
    df[master] = df[master].fillna(master_median)
    print(f"Master '{master}' missing values filled with median: {master_median}")
    
    for sub in sub_cols:
        user_median = df[df[master] == 1][sub].median()
        
        df.loc[df[master] == 0, sub] = df.loc[df[master] == 0, sub].fillna(0)
        df.loc[df[master] == 1, sub] = df.loc[df[master] == 1, sub].fillna(user_median)
        
        print(f"  -> '{sub}' conditionally filled (0 if {master}=0, else {user_median})")
print("\n")

# ---------------------------------------------------------
# 3. K-Nearest Neighbors (KNN) Imputation
# ---------------------------------------------------------
print("--- K-Nearest Neighbors (KNN) Imputation Log ---")

# Identify binary columns to snap fractional predictions back to 0 or 1 later
binary_cols = [col for col in df.columns if set(df[col].dropna().unique()).issubset({0.0, 1.0})]

# Save a boolean mask of where the missing values originally were
missing_mask = df.isnull()

# A. Scale the data so large numerical values do not dominate the distance calculation
scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns, index=df.index)

# B. Execute KNN Imputation (k=5)
imputer = KNNImputer(n_neighbors=5, weights='distance')
df_imputed_scaled = pd.DataFrame(imputer.fit_transform(df_scaled), columns=df.columns, index=df.index)

# C. Inverse transform back to original feature magnitudes
df = pd.DataFrame(scaler.inverse_transform(df_imputed_scaled), columns=df.columns, index=df.index)

# D. Snap binary columns back to absolute bounds
for col in binary_cols:
    if missing_mask[col].any():
        # Standard rounding: < 0.5 becomes 0.0, >= 0.5 becomes 1.0
        df[col] = np.round(df[col])

# Report KNN changes
knn_imputed_count = 0
for col in df.columns:
    missing_count = missing_mask[col].sum()
    if missing_count > 0:
        print(f"KNN Imputed {missing_count} missing values in '{col}'.")
        knn_imputed_count += missing_count

print(f"Total isolated values imputed via KNN: {knn_imputed_count}\n")

# ---------------------------------------------------------
# 4. Post-Imputation Consistency Enforcement
# ---------------------------------------------------------
print("--- Post-Imputation Consistency Check & Override ---")
corrections_made = 0

all_std_related_cols = [col for col in df.columns if col.startswith('STDs') and col not in ['STDs', 'STDs (number)']]

# Rule 1: If KNN determined any specific STD is present (>0), the master STDs flag MUST be 1.0
conflict_mask_1 = (df['STDs'] == 0) & (df[all_std_related_cols].sum(axis=1) > 0)
num_conflicts_1 = conflict_mask_1.sum()
if num_conflicts_1 > 0:
    df.loc[conflict_mask_1, 'STDs'] = 1.0
    print(f"WARNING: KNN created {num_conflicts_1} contradictions where sub-STDs existed but master STDs=0.")
    print("-> OVERRIDE APPLIED: Master 'STDs' forced to 1.0 for these rows.")
    corrections_made += num_conflicts_1

# Rule 2: If KNN determined the master STDs flag is 0, ALL specific STDs must mathematically be 0.0
# We iterate through the sub-columns and crush any fractional bleed-over or erroneous positive flags
conflict_mask_2 = (df['STDs'] == 0) & (df[all_std_related_cols].sum(axis=1) > 0)
num_conflicts_2 = (df.loc[df['STDs'] == 0, all_std_related_cols] > 0).sum().sum()
if num_conflicts_2 > 0:
    df.loc[df['STDs'] == 0, all_std_related_cols] = 0.0
    print(f"WARNING: KNN created {num_conflicts_2} contradictions where master STDs=0 but sub-STDs were flagged.")
    print("-> OVERRIDE APPLIED: All sub-STDs forced to 0.0 for these rows.")
    corrections_made += num_conflicts_2

# Same logic for the Condylomatosis hierarchy
condy_cols = [col for col in df.columns if 'condylomatosis' in col and col != 'STDs:condylomatosis']
condy_conflict = (df['STDs:condylomatosis'] == 0) & (df[condy_cols].sum(axis=1) > 0)
num_condy = condy_conflict.sum()
if num_condy > 0:
    df.loc[condy_conflict, 'STDs:condylomatosis'] = 1.0
    print(f"WARNING: KNN created {num_condy} contradictions in Condylomatosis hierarchy.")
    print("-> OVERRIDE APPLIED: Master 'STDs:condylomatosis' forced to 1.0.")
    corrections_made += num_condy

if corrections_made == 0:
    print("SUCCESS: KNN imputation generated zero logical contradictions.")
else:
    print(f"Total programmatic overrides executed: {corrections_made}")
print("\n")

# ---------------------------------------------------------
# 5. Final Proof
# ---------------------------------------------------------
print("--- Final Dataset Check ---")
print(f"Total missing values left in the ENTIRE dataset: {df.isnull().sum().sum()}")
print(f"Final Dataset Shape: {df.shape[0]} patients, {df.shape[1]} features")
print("==========================================\n")

PHASE 4: HANDLING MISSING DATA (IMPUTATION)
Dropped 2 columns due to >90% missing data.

--- Deterministic Conditional Imputation Log ---
Master 'IUD' missing values filled with median: 0.0
  -> 'IUD (years)' conditionally filled (0 if IUD=0, else 4.0)
Master 'Hormonal Contraceptives' missing values filled with median: 1.0
  -> 'Hormonal Contraceptives (years)' conditionally filled (0 if Hormonal Contraceptives=0, else 2.0)
Master 'Smokes' missing values filled with median: 0.0
  -> 'Smokes (years)' conditionally filled (0 if Smokes=0, else 7.0)
  -> 'Smokes (packs/year)' conditionally filled (0 if Smokes=0, else 1.2)


--- K-Nearest Neighbors (KNN) Imputation Log ---
KNN Imputed 26 missing values in 'Number of sexual partners'.
KNN Imputed 7 missing values in 'First sexual intercourse'.
KNN Imputed 55 missing values in 'Num of pregnancies'.
KNN Imputed 105 missing values in 'STDs'.
KNN Imputed 105 missing values in 'STDs (number)'.
KNN Imputed 105 missing values in 'STDs:condylomatosi

# Phase 5: Export clean data

In [54]:
# Phase 5: Export Clean Data
# Save the cleaned dataset so we can pick it up in the next notebook
output_path = "../data/clean_cervical_cancer_data.csv"
df.to_csv(output_path, index=False)

print(f"Clean data successfully saved to {output_path}")

Clean data successfully saved to ../data/clean_cervical_cancer_data.csv
